# Smoothing with Moving Averages: Complete Guide

## 1. The Philosophy of Smoothing: Signal Extraction
- In time series analysis, we operate under the assumption that the observed data (Y_t) is a composite: a mix of the true "signal" and random "noise".
- Smoothing acts as a low-pass filter. It works by suppressing high-frequency oscillations (the noise) while preserving the low-frequency components (the trend and cyclical patterns).
- The choice of smoothing method is a trade-off between **smoothness** (how much noise is removed) and **fidelity** (how much of the original trend information is retained).
- In this notebook, we will explore the mathematical mechanics, business implications, and Python implementations of various moving average techniques.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)

print("Libraries successfully imported! Environment is ready.")

## 3. Data Creation: Synthetic Pharmaceutical Sales
- To demonstrate smoothing, we need a dataset with a clear underlying trend, some seasonality, and significant daily/weekly noise.
- We will simulate Daily Pharmaceutical Sales spanning roughly 6 months (180 days).
- The dataset will have a baseline growth (trend), a weekly cycle (higher sales on weekdays, lower on weekends), and random volatility (noise).

In [ ]:
# Generate temporal index (180 days)
date_rng = pd.date_range(start='2026-01-01', periods=180, freq='D')
time = np.arange(180)

# 1. Trend Component (Steady upward growth)
trend = 200 + (1.5 * time)

# 2. Seasonal Component (Weekly cycle: 7-day period)
# We use a sine wave that peaks mid-week and drops on weekends
seasonality = 50 * np.sin(2 * np.pi * time / 7)

# 3. Noise Component (High volatility day-to-day)
noise = np.random.normal(loc=0, scale=40, size=180)

# Combine components
daily_sales = trend + seasonality + noise

# Create DataFrame
df_pharma = pd.DataFrame({
    'Date': date_rng,
    'Sales': daily_sales,
    'True_Signal': trend + seasonality
})

df_pharma.set_index('Date', inplace=True)

print("Synthetic Pharmaceutical Sales dataset created with 180 records.")
print("\n--- First 5 rows of the dataset ---")
print(df_pharma.head().round(2))
print("\n--- Descriptive Statistics ---")
print(df_pharma.describe().round(2))

## 4. Visualizing the Raw Data (Signal vs. Noise)
- Before applying any mathematical smoothing, we must visually inspect the data.
- The "True Signal" represents what we are trying to extract, but in reality, we only observe the "Raw Sales" (Signal + Noise).

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df_pharma.index, df_pharma['Sales'], color='lightgray', marker='o', markersize=4, linestyle='-', label='Raw Sales (Observed)')
plt.plot(df_pharma.index, df_pharma['True_Signal'], color='blue', linewidth=2, label='True Signal (Hidden)')

plt.title('Raw Sales Data vs. Hidden True Signal', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: The gray line is highly erratic. It is difficult for management to assess the growth trend purely from the raw numbers.")

## 5. The Mechanics of the Simple Moving Average (SMA)
- The Simple Moving Average is the workhorse of smoothing. It operates on the principle of a **rolling window**.
- For a window of size k, the smoothed value Y_hat_t is calculated as the arithmetic mean of the current observation and the previous (k-1) observations:
- Formula: Y_hat_t = (1/k) * Sum(Y_{t-i}) from i=0 to k-1
- This is an "equal-weight" model. It assumes data from k periods ago is just as important as yesterday's data.

In [ ]:
# Calculate a 7-Day and 14-Day Trailing Simple Moving Average
# We use rolling(window=k).mean() to achieve this in pandas
df_pharma['SMA_7'] = df_pharma['Sales'].rolling(window=7).mean()
df_pharma['SMA_14'] = df_pharma['Sales'].rolling(window=14).mean()

print("--- Data with Trailing SMAs ---")
print(df_pharma[['Sales', 'SMA_7', 'SMA_14']].head(15).round(2))

print("\nNotice the NaN (Not a Number) values at the start.")
print("A trailing window of size k requires k observations to compute the first value.")
print("This is the first drawback of moving averages: boundary data loss.")

## 6. The Window Size (k): Balancing Fidelity and Lag
- The parameter k is your tuning knob. It controls the Bias-Variance trade-off in time series smoothing.
- **Small k (e.g., 7)**: Low bias, high variance. Tracks quick changes but retains too much noise.
- **Large k (e.g., 30)**: High bias, low variance. Very smooth, but introduces massive **Lag**. It "smooths out" actual trend reversals.

In [ ]:
df_pharma['SMA_30'] = df_pharma['Sales'].rolling(window=30).mean()

plt.figure(figsize=(14, 6))
plt.plot(df_pharma.index, df_pharma['Sales'], color='lightgray', alpha=0.6, label='Raw Sales')
plt.plot(df_pharma.index, df_pharma['SMA_7'], color='green', linewidth=2, label='SMA (k=7)')
plt.plot(df_pharma.index, df_pharma['SMA_30'], color='red', linewidth=2, label='SMA (k=30)')

plt.title('The Impact of Window Size (k) and The Lag Problem', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation on Lag: Look closely at the red line (k=30).")
print("Because it averages the past 30 days, it continuously 'chases' the actual trend but never catches up.")
print("In a rapidly growing business, a large trailing SMA will consistently underestimate current performance.")

## 7. Centered Moving Average (CMA)
- If you are analyzing historical data and do not need a real-time forecast, you use a **Centered Moving Average**.
- This averages observations before and after the current point t, effectively eliminating the lag issue entirely.
- Formula for odd k: Y_hat_t = 1/(k) * Sum(Y_{t+i}) from i = -(k-1)/2 to (k-1)/2
- Drawback: Data loss at BOTH ends of the series. You cannot center an average on today's date because you don't have tomorrow's data.

In [ ]:
# Calculate a 15-Day Centered Moving Average
# In pandas, setting center=True shifts the window to align with the middle index
df_pharma['CMA_15'] = df_pharma['Sales'].rolling(window=15, center=True).mean()

plt.figure(figsize=(14, 6))
plt.plot(df_pharma.index, df_pharma['True_Signal'], color='blue', linewidth=2, alpha=0.5, label='True Signal')
plt.plot(df_pharma.index, df_pharma['SMA_14'], color='red', linestyle='--', linewidth=2, label='Trailing SMA (Lagging)')
plt.plot(df_pharma.index, df_pharma['CMA_15'], color='darkorange', linewidth=2.5, label='Centered MA (No Lag)')

plt.title('Fixing Lag with Centered Moving Averages', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: The orange CMA perfectly aligns with the phase of the true signal. It does not lag.")
print("However, check the tail end of the data... the CMA stops before the series ends because it lacks 'future' data points.")

## 8. Weighted Moving Average (WMA)
- The "Equal Weight" Fallacy: SMA assumes data from 10 days ago is as relevant as yesterday's data.
- Weighted Moving Average (WMA) assigns higher weights to more recent observations. This allows the trend to be more responsive to recent shifts while still filtering out noise.

In [ ]:
# Define a custom function to apply linear weights
def calculate_wma(window_data):
    # Create weights: [1, 2, 3, ... k]
    weights = np.arange(1, len(window_data) + 1)
    # Calculate weighted average: sum(data * weights) / sum(weights)
    return np.dot(window_data, weights) / weights.sum()

# Apply a 14-day WMA
df_pharma['WMA_14'] = df_pharma['Sales'].rolling(window=14).apply(calculate_wma, raw=True)

plt.figure(figsize=(14, 6))
plt.plot(df_pharma.index[-50:], df_pharma['Sales'].iloc[-50:], color='lightgray', marker='o', label='Raw Sales (Last 50 Days)')
plt.plot(df_pharma.index[-50:], df_pharma['SMA_14'].iloc[-50:], color='red', label='SMA (Equal Weight)')
plt.plot(df_pharma.index[-50:], df_pharma['WMA_14'].iloc[-50:], color='purple', label='WMA (Recent Heavy)')

plt.title('SMA vs. WMA: Responsiveness to Recent Data', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: The purple WMA tracks closer to the actual recent data points because yesterday's sales count 14x more than the sales from two weeks ago.")

## 9. Exponential Smoothing (EMA/ETS)
- When moving averages are too restrictive, analysts turn to Exponential Smoothing.
- It is a recursive approach where the weight of older observations decreases exponentially as we go back in time.
- Formula: Y_hat_t = alpha * Y_t + (1 - alpha) * Y_hat_{t-1}
- alpha is the smoothing factor (between 0 and 1). 
- A high alpha makes the model responsive (closer to raw data), while a low alpha makes it extremely smooth.
- Crucial Benefit: No boundary data loss! It starts calculating immediately.

In [ ]:
# Calculate Exponential Smoothing using pandas ewm() (Exponential Weighted Math)
# We will test a slow adapter (alpha=0.1) and a fast adapter (alpha=0.5)
df_pharma['EMA_0.1'] = df_pharma['Sales'].ewm(alpha=0.1, adjust=False).mean()
df_pharma['EMA_0.5'] = df_pharma['Sales'].ewm(alpha=0.5, adjust=False).mean()

plt.figure(figsize=(14, 6))
plt.plot(df_pharma.index, df_pharma['Sales'], color='lightgray', alpha=0.6, label='Raw Sales')
plt.plot(df_pharma.index, df_pharma['EMA_0.5'], color='teal', linewidth=1.5, label='EMA (alpha=0.5) - Fast')
plt.plot(df_pharma.index, df_pharma['EMA_0.1'], color='navy', linewidth=2.5, label='EMA (alpha=0.1) - Slow')

plt.title('Exponential Smoothing: The Power of Alpha', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: EMA has no 'Null' gaps at the beginning of the chart. It solves the End-of-Series Blind Spot.")

## 10. Business Application: Anomaly Detection using Smoothed Baselines
- Smoothing is not just a data transformation; it is a management tool.
- Once the smoothed trend is established, you can use it as a baseline to identify anomalies.
- If a raw data point deviates significantly from the moving average (e.g., > 2 standard deviations), it acts as a 'trigger' for investigation.

In [ ]:
# 1. Inject an artificial anomaly into our data (e.g., a massive supply chain failure)
df_pharma.loc['2026-04-15', 'Sales'] = 100  # Normal sales would be ~350

# 2. Calculate a 14-day trailing mean and standard deviation
rolling_window = df_pharma['Sales'].rolling(window=14)
df_pharma['Baseline'] = rolling_window.mean()
df_pharma['Vol_Std'] = rolling_window.std()

# 3. Create Upper and Lower control bounds (Baseline +/- 2 Std Dev)
df_pharma['Upper_Bound'] = df_pharma['Baseline'] + (2 * df_pharma['Vol_Std'])
df_pharma['Lower_Bound'] = df_pharma['Baseline'] - (2 * df_pharma['Vol_Std'])

# 4. Identify the anomaly
anomalies = df_pharma[df_pharma['Sales'] < df_pharma['Lower_Bound']]

plt.figure(figsize=(14, 6))
plt.plot(df_pharma.index, df_pharma['Sales'], color='gray', label='Sales')
plt.plot(df_pharma.index, df_pharma['Baseline'], color='blue', label='Smoothed Baseline (14d)')
plt.fill_between(df_pharma.index, df_pharma['Lower_Bound'], df_pharma['Upper_Bound'], color='blue', alpha=0.1, label='Expected Range (95% confidence)')
plt.scatter(anomalies.index, anomalies['Sales'], color='red', s=100, zorder=5, label='Detected Anomaly')

plt.title('Using Moving Averages for Business Anomaly Detection', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"System flagged {len(anomalies)} anomaly(ies). Management alert generated for Date: {anomalies.index[0].date()}")

## 11. Practice Exercises: Server Traffic Analysis
**Scenario:** You are analyzing daily server traffic for a SaaS application. The traffic has heavy weekly seasonality and some noise.
Let's generate the practice data.

In [ ]:
# Generate practice data
n_days = 90
dates_web = pd.date_range(start='2026-07-01', periods=n_days, freq='D')
time_web = np.arange(n_days)

web_trend = 1000 + (5 * time_web)
web_season = 300 * np.sin(2 * np.pi * time_web / 7)
web_noise = np.random.normal(0, 150, n_days)

df_web = pd.DataFrame({
    'Date': dates_web,
    'Traffic': web_trend + web_season + web_noise
})
df_web.set_index('Date', inplace=True)
print("Practice Web Traffic Dataset Created.")
print(df_web.head())

### Exercise 1: Implementing a 7-Day Centered Moving Average
- **Task:** Calculate a 7-day Centered Moving Average to extract the underlying trend, removing the weekly seasonality perfectly without introducing lag.
- Store it in a column called `CMA_7`.
- Plot the raw traffic and the CMA_7.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
df_web['CMA_7'] = df_web['Traffic'].rolling(window=7, center=True).mean()

plt.figure(figsize=(12, 5))
plt.plot(df_web.index, df_web['Traffic'], color='lightgray', marker='.', label='Raw Web Traffic')
plt.plot(df_web.index, df_web['CMA_7'], color='green', linewidth=3, label='7-Day Centered MA')
plt.title('Extracting Trend with Centered Moving Average', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

print("Notice how a 7-day window completely nullifies a 7-day seasonal cycle. The green line represents pure trend.")

### Exercise 2: Implementing Exponential Smoothing
- **Task:** The networking team needs real-time alerting without losing data points at the boundary.
- Calculate an Exponential Smoothing line with an alpha of 0.25.
- Store it in a column called `EMA_0.25` and display the first 10 rows.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
df_web['EMA_0.25'] = df_web['Traffic'].ewm(alpha=0.25, adjust=False).mean()

print("--- First 10 days of Web Traffic with EMA ---")
print(df_web[['Traffic', 'EMA_0.25']].head(10).round(1))

print("\nSuccess: The EMA starts calculating on Day 1, avoiding the NaN issue of Simple Moving Averages.")

## 12. Summary and Key Takeaways

1. **Signal vs. Noise**: Smoothing helps decouple the true underlying trend from day-to-day volatility.
2. **Simple Moving Average (SMA)**: An equal-weight rolling window. Easy to understand, but suffers from boundary data loss and introduces **Lag**.
3. **The Parameter 'k'**: The window size is a bias-variance trade-off. Small k is noisy but responsive. Large k is smooth but heavily lagged.
4. **Centered Moving Averages (CMA)**: Fixes the lag problem for historical data but causes data loss at the most recent edge (useless for forecasting tomorrow).
5. **Weighted Moving Averages (WMA)**: Overcomes the "equal-weight fallacy" by prioritizing recent data points.
6. **Exponential Smoothing (EMA/ETS)**: The modern standard. It applies exponentially decaying weights to the past, preventing boundary data loss and allowing for adaptive responsiveness via the alpha parameter.

In [ ]:
print("Module 'Smoothing with Moving Averages' completed successfully.")
print("Smoothing is the foundational step before advanced forecasting models like SARIMA and Prophet.")